In [1]:
# ============================================================
# CELL 1 — Environment and source data provenance
# ============================================================
# Verify the research environment and original source files
# before processing any measurement data.

from pathlib import Path
import hashlib
import sys
import platform

# Locate the project root independently of notebook location.
ROOT = next(
    (
        path for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "PROTOCOL.md").is_file()
        and (path / "data" / "SHA256SUMS").is_file()
    ),
    None
)

assert ROOT is not None, "Research project root not found."

DATA = ROOT / "data"
RAW = DATA / "raw"

# Verify that the notebook uses the isolated project environment.
assert Path(sys.prefix).resolve() == (ROOT / ".venv").resolve(), (
    "Incorrect Python environment. Select the project kernel."
)

# Read the recorded source provenance.
SOURCE_URL = (DATA / "source_url.txt").read_text().strip()
SOURCE_COMMIT = (DATA / "source_commit.txt").read_text().strip()

# Load the recorded SHA-256 manifest.
manifest = {}

for line in (DATA / "SHA256SUMS").read_text().splitlines():
    expected_hash, relative_path = line.split(maxsplit=1)
    manifest[relative_path] = expected_hash

# Verify that the actual file inventory matches the manifest.
actual_files = {
    path.relative_to(ROOT).as_posix()
    for path in RAW.rglob("*")
    if path.is_file()
}

assert actual_files == set(manifest), "File inventory mismatch."

# Independently verify every original file.
print("=== FILE INTEGRITY ===")

verified_files = 0

for relative_path, expected_hash in sorted(manifest.items()):
    actual_hash = hashlib.sha256(
        (ROOT / relative_path).read_bytes()
    ).hexdigest()

    assert actual_hash == expected_hash, (
        f"SHA-256 mismatch: {relative_path}"
    )

    verified_files += 1
    print(f"PASS | {relative_path}")

# Report the verified research environment.
print("\n=== RESEARCH ENVIRONMENT ===")
print("Python:", platform.python_version())
print("Platform:", platform.platform())
print("Architecture:", platform.machine())
print("Environment:", sys.prefix)
print("Project root:", ROOT)

print("\n=== DATA PROVENANCE ===")
print("Source:", SOURCE_URL)
print("Source commit:", SOURCE_COMMIT)
print("Verified files:", verified_files)
print("Manifest entries:", len(manifest))

print("\nVALIDATION: PASS")

=== FILE INTEGRITY ===
PASS | data/raw/Codes/logger.py
PASS | data/raw/Codes/smart_contract.sol
PASS | data/raw/LICENSE
PASS | data/raw/Meta Data/proof_log.json
PASS | data/raw/Meta Data/unsubmitted_records.json
PASS | data/raw/logs/energy_log_2025-06-12.csv
PASS | data/raw/logs/energy_log_2025-06-13.csv
PASS | data/raw/logs/energy_log_2025-06-14.csv
PASS | data/raw/logs/energy_log_2025-06-15.csv
PASS | data/raw/logs/energy_log_2025-06-16.csv
PASS | data/raw/logs/energy_log_2025-06-17.csv
PASS | data/raw/logs/energy_log_2025-06-18.csv

=== RESEARCH ENVIRONMENT ===
Python: 3.12.2
Platform: macOS-26.2-arm64-arm-64bit
Architecture: arm64
Environment: /Users/itsjayhani/Research/iot-evidence-assurance-benchmark/.venv
Project root: /Users/itsjayhani/Research/iot-evidence-assurance-benchmark

=== DATA PROVENANCE ===
Source: https://github.com/javadvf/solar_logger.git
Source commit: ec34ea5dc21642744d39eb436b787ac092505d0f
Verified files: 12
Manifest entries: 12

VALIDATION: PASS


In [2]:
# ============================================================
# CELL 2 — Raw measurement structure and quality audit
# ============================================================

import csv
import re
from datetime import datetime
from decimal import Decimal, InvalidOperation

# Expected fields from the original logger and CSV schema.
EXPECTED_FIELDS = (
    "timestamp",
    "voltage",
    "current",
    "power",
    "energy_Wh",
    "cumulative_energy_Wh",
    "device_id",
    "chain_hash",
)

NUMERIC_FIELDS = EXPECTED_FIELDS[1:6]

csv_files = sorted((RAW / "logs").glob("*.csv"))

assert csv_files, "No measurement CSV files found."

# Preserve original string values for cryptographic reproduction.
records_by_file = {}
audit_errors = []

print("=== RAW MEASUREMENT AUDIT ===\n")

for file in csv_files:

    with file.open("r", encoding="utf-8", newline="") as stream:
        reader = csv.DictReader(stream, strict=True)

        assert tuple(reader.fieldnames or ()) == EXPECTED_FIELDS, (
            f"Unexpected CSV schema: {file.name}"
        )

        records = list(reader)

    records_by_file[file.name] = records

    # Validate structure and data types without changing values.
    for line_number, record in enumerate(records, start=2):

        try:
            if None in record or any(
                value is None or value == ""
                for value in record.values()
            ):
                raise ValueError("Missing or unexpected fields")

            datetime.fromisoformat(record["timestamp"])

            for field in NUMERIC_FIELDS:
                if not Decimal(record[field]).is_finite():
                    raise ValueError(f"Non-finite value: {field}")

            if not record["device_id"].strip():
                raise ValueError("Empty device identifier")

            if not re.fullmatch(
                r"[0-9a-f]{64}", record["chain_hash"]
            ):
                raise ValueError("Malformed SHA-256 hash")

        except (ValueError, InvalidOperation) as error:
            audit_errors.append(
                (file.name, line_number, str(error))
            )

    print(f"{file.name}: {len(records):,} rows")

# Combine records without removing duplicates.
raw_records = [
    record
    for records in records_by_file.values()
    for record in records
]

print("\n=== DATASET SUMMARY ===")
print("CSV files:", len(csv_files))
print("Raw measurement rows:", len(raw_records))
print("Fields per record:", len(EXPECTED_FIELDS))
print("Device identifiers:", sorted({
    record["device_id"] for record in raw_records
}))
print("Validation errors:", len(audit_errors))

if audit_errors:
    print("\nFirst five errors:")
    for error in audit_errors[:5]:
        print(error)

assert not audit_errors, "Raw data validation failed."

print("\nVALIDATION: PASS")

=== RAW MEASUREMENT AUDIT ===

energy_log_2025-06-12.csv: 328 rows
energy_log_2025-06-13.csv: 1,768 rows
energy_log_2025-06-14.csv: 1,794 rows
energy_log_2025-06-15.csv: 1,799 rows
energy_log_2025-06-16.csv: 1,816 rows
energy_log_2025-06-17.csv: 1,848 rows
energy_log_2025-06-18.csv: 915 rows

=== DATASET SUMMARY ===
CSV files: 7
Raw measurement rows: 10268
Fields per record: 8
Device identifiers: ['device_001']
Validation errors: 0

VALIDATION: PASS


In [3]:
# ============================================================
# CELL 3 — Deduplication and temporal analysis
# ============================================================

from collections import Counter, defaultdict
from statistics import median

# Identify exact duplicates using all original fields.
record_groups = defaultdict(list)

for filename, records in records_by_file.items():
    for record in records:
        key = tuple(record[field] for field in EXPECTED_FIELDS)
        record_groups[key].append(filename)

# Preserve one copy of each distinct measurement record.
unique_records = [
    dict(zip(EXPECTED_FIELDS, key))
    for key in record_groups
]

# Sort records chronologically without changing their values.
unique_records.sort(
    key=lambda record: datetime.fromisoformat(record["timestamp"])
)

duplicate_count = len(raw_records) - len(unique_records)

# Identify conflicting records with identical device timestamps.
timestamp_counts = Counter(
    (record["device_id"], record["timestamp"])
    for record in unique_records
)

timestamp_conflicts = {
    key: count
    for key, count in timestamp_counts.items()
    if count > 1
}

# Check whether distinct records share the same chain hash.
hash_counts = Counter(
    record["chain_hash"] for record in unique_records
)

repeated_hashes = {
    key: count
    for key, count in hash_counts.items()
    if count > 1
}

# Investigate whether duplicate records occur across files.
cross_file_duplicates = sum(
    len(files) - len(set(files))
    == 0 and len(files) > 1
    for files in record_groups.values()
)

within_file_duplicate_rows = sum(
    len(files) - len(set(files))
    for files in record_groups.values()
)

# Calculate sampling intervals separately for each device.
timestamps_by_device = defaultdict(list)

for record in unique_records:
    timestamps_by_device[record["device_id"]].append(
        datetime.fromisoformat(record["timestamp"])
    )

sampling_intervals = [
    (current - previous).total_seconds()
    for timestamps in timestamps_by_device.values()
    for previous, current in zip(timestamps, timestamps[1:])
]

assert sampling_intervals, "Insufficient temporal observations."

# Report descriptive results without arbitrary gap thresholds.
print("=== DEDUPLICATION ===")
print("Raw rows:", len(raw_records))
print("Unique records:", len(unique_records))
print("Duplicate rows:", duplicate_count)
print(f"Duplicate percentage: {duplicate_count / len(raw_records):.4%}")

print("\n=== DUPLICATION STRUCTURE ===")
print("Distinct records repeated across files:", cross_file_duplicates)
print("Within-file duplicate rows:", within_file_duplicate_rows)
print("Conflicting device timestamps:", len(timestamp_conflicts))
print("Repeated hashes among distinct records:", len(repeated_hashes))

print("\n=== TEMPORAL ANALYSIS ===")
print("First timestamp:", unique_records[0]["timestamp"])
print("Last timestamp:", unique_records[-1]["timestamp"])

duration = (
    datetime.fromisoformat(unique_records[-1]["timestamp"])
    - datetime.fromisoformat(unique_records[0]["timestamp"])
).total_seconds()

print(f"Observation duration: {duration / 3600:.6f} hours")
print("Sampling intervals:", len(sampling_intervals))
print(f"Minimum interval: {min(sampling_intervals):.6f} seconds")
print(f"Median interval: {median(sampling_intervals):.6f} seconds")
print(f"Maximum interval: {max(sampling_intervals):.6f} seconds")

# Validate record accounting and chronological consistency.
assert len(raw_records) == len(unique_records) + duplicate_count
assert not timestamp_conflicts, "Conflicting timestamps detected."
assert not repeated_hashes, "Repeated chain hashes detected."
assert all(interval > 0 for interval in sampling_intervals)

print("\nVALIDATION: PASS")

=== DEDUPLICATION ===
Raw rows: 10268
Unique records: 8106
Duplicate rows: 2162
Duplicate percentage: 21.0557%

=== DUPLICATION STRUCTURE ===
Distinct records repeated across files: 2162
Within-file duplicate rows: 0
Conflicting device timestamps: 0
Repeated hashes among distinct records: 0

=== TEMPORAL ANALYSIS ===
First timestamp: 2025-06-12T18:32:12.452010
Last timestamp: 2025-06-18T09:37:13.542316
Observation duration: 135.083636 hours
Sampling intervals: 8105
Minimum interval: 59.966479 seconds
Median interval: 60.000132 seconds
Maximum interval: 60.034032 seconds

VALIDATION: PASS


In [4]:
# ============================================================
# CELL 4 — Proof metadata and batch coverage audit
# ============================================================

import json

# Load the original proof metadata and unsubmitted records.
metadata_dir = RAW / "Meta Data"

with (metadata_dir / "proof_log.json").open() as stream:
    proofs = json.load(stream)

with (metadata_dir / "unsubmitted_records.json").open() as stream:
    pending_records = json.load(stream)

assert isinstance(proofs, list) and proofs
assert isinstance(pending_records, list)

# Reconstruct batches using the original CSV row boundaries.
# Preserve the original row order and textual field values.
batches = []
covered_hashes = []
transaction_hashes = []

for proof in proofs:

    filename = Path(proof["csv_path"]).name

    assert filename in records_by_file, (
        f"Unknown source file: {filename}"
    )

    start = proof["row_start"]
    end = proof["row_end"]
    size = proof["size"]

    assert all(type(value) is int for value in (start, end, size))
    assert 0 <= start <= end < len(records_by_file[filename])
    assert end - start + 1 == size

    # Validate cryptographic metadata formatting.
    for field in ("root", "batch_hash", "tx_hash"):
        assert re.fullmatch(
            r"0x[0-9a-fA-F]{64}", proof[field]
        ), f"Malformed {field}"

    assert re.fullmatch(
        r"[0-9a-fA-F]{64}", proof["csv_sha256"]
    )

    datetime.fromisoformat(proof["timestamp"])

    batch_records = records_by_file[filename][start:end + 1]

    batches.append({
        "proof": proof,
        "records": batch_records,
    })

    covered_hashes.extend(
        record["chain_hash"] for record in batch_records
    )

    transaction_hashes.append(proof["tx_hash"].lower())

# Check coverage and identify overlapping batch references.
coverage_counts = Counter(covered_hashes)

overlapping_references = sum(
    count - 1
    for count in coverage_counts.values()
)

unique_hashes = {
    record["chain_hash"] for record in unique_records
}

covered_hash_set = set(covered_hashes)

unreferenced_records = [
    record for record in unique_records
    if record["chain_hash"] not in covered_hash_set
]

unreferenced_hashes = {
    record["chain_hash"] for record in unreferenced_records
}

pending_hashes = {
    record["chain_hash"] for record in pending_records
}

# Check whether the batches follow the measurement chronology.
chronological_hashes = [
    record["chain_hash"] for record in unique_records
]

chronological_coverage = (
    covered_hashes
    == chronological_hashes[:len(covered_hashes)]
)

batch_sizes = [
    len(batch["records"]) for batch in batches
]

# Report audit results.
print("=== PROOF METADATA ===")
print("Proof entries:", len(proofs))
print("Reconstructed batches:", len(batches))
print("Unique transaction hashes:", len(set(transaction_hashes)))
print("Invalid batch boundaries: 0")

print("\n=== BATCH STRUCTURE ===")
print("Minimum batch size:", min(batch_sizes))
print("Median batch size:", median(batch_sizes))
print("Maximum batch size:", max(batch_sizes))
print("Total batch record references:", len(covered_hashes))
print("Overlapping batch references:", overlapping_references)
print("Chronological coverage:", chronological_coverage)

print("\n=== RECORD COVERAGE ===")
print("Unique measurements:", len(unique_records))
print("Records referenced by proofs:", len(covered_hash_set))
print("Unreferenced records:", len(unreferenced_records))
print("Records in pending queue:", len(pending_records))
print("Unreferenced records found in queue:",
      len(unreferenced_hashes & pending_hashes))

print(f"Proof metadata coverage: "
      f"{len(covered_hash_set) / len(unique_records):.4%}")

if unreferenced_records:
    print("\n=== UNREFERENCED PERIOD ===")
    print("First:", unreferenced_records[0]["timestamp"])
    print("Last:", unreferenced_records[-1]["timestamp"])

# Validate the reconstructed dataset.
assert len(set(transaction_hashes)) == len(proofs)
assert overlapping_references == 0
assert covered_hash_set <= unique_hashes
assert chronological_coverage
assert unreferenced_hashes == pending_hashes
assert len(pending_records) == len(pending_hashes)

assert (
    len(covered_hash_set) + len(unreferenced_records)
    == len(unique_records)
)

print("\nVALIDATION: PASS")

=== PROOF METADATA ===
Proof entries: 130
Reconstructed batches: 130
Unique transaction hashes: 130
Invalid batch boundaries: 0

=== BATCH STRUCTURE ===
Minimum batch size: 24
Median batch size: 34.0
Maximum batch size: 695
Total batch record references: 8085
Overlapping batch references: 0
Chronological coverage: True

=== RECORD COVERAGE ===
Unique measurements: 8106
Records referenced by proofs: 8085
Unreferenced records: 21
Records in pending queue: 21
Unreferenced records found in queue: 21
Proof metadata coverage: 99.7409%

=== UNREFERENCED PERIOD ===
First: 2025-06-18T09:17:13.539805
Last: 2025-06-18T09:37:13.542316

VALIDATION: PASS


In [5]:
# ============================================================
# CELL 5 — Record-level chained hash reproduction
# ============================================================
# Independently reproduce the original SHA-256 hash chain.
# Preserve the original field order and textual representation.
#
# Source: original solar_logger/Codes/logger.py
# SHA-256 reference: NIST FIPS 180-4

HASH_FIELDS = EXPECTED_FIELDS[:-1]

# Verify SHA-256 against a known reference test vector.
assert hashlib.sha256(b"abc").hexdigest() == (
    "ba7816bf8f01cfea414140de5dae2223"
    "b00361a396177a9cb410ff61f20015ad"
)

# Original logger initialization: 64 hexadecimal zeroes.
INITIAL_HASH = "0" * 64

previous_hash = INITIAL_HASH
reproduced_hashes = []
chain_mismatches = []

for index, record in enumerate(unique_records):

    # Reproduce the original comma-separated serialization.
    content = ",".join(
        record[field] for field in HASH_FIELDS
    )

    # Independently calculate the next chained hash.
    chained_input = previous_hash + content

    computed_hash = hashlib.sha256(
        chained_input.encode("utf-8")
    ).hexdigest()

    reproduced_hashes.append(computed_hash)

    # Compare the calculated hash with the stored hash.
    if computed_hash != record["chain_hash"]:
        chain_mismatches.append({
            "index": index,
            "timestamp": record["timestamp"],
            "expected": record["chain_hash"],
            "computed": computed_hash,
        })

    # Continue using the independently calculated hash.
    previous_hash = computed_hash

# Report verification results.
total_checked = len(reproduced_hashes)
matched_hashes = total_checked - len(chain_mismatches)

print("=== SHA-256 IMPLEMENTATION CHECK ===")
print("Reference test vector: PASS")

print("\n=== CHAIN HASH REPRODUCTION ===")
print("Records checked:", total_checked)
print("Matching hashes:", matched_hashes)
print("Mismatched hashes:", len(chain_mismatches))
print(f"Reproduction rate: {matched_hashes / total_checked:.4%}")

print("\n=== CHAIN BOUNDARIES ===")
print("Initial seed:", INITIAL_HASH)
print("First reproduced hash:", reproduced_hashes[0])
print("Last reproduced hash:", reproduced_hashes[-1])
print("Last stored hash:", unique_records[-1]["chain_hash"])

if chain_mismatches:
    print("\nFirst five mismatches:")
    for mismatch in chain_mismatches[:5]:
        print(mismatch)

assert total_checked == len(unique_records)
assert not chain_mismatches, "Hash chain reproduction failed."

print("\nVALIDATION: PASS")

=== SHA-256 IMPLEMENTATION CHECK ===
Reference test vector: PASS

=== CHAIN HASH REPRODUCTION ===
Records checked: 8106
Matching hashes: 8106
Mismatched hashes: 0
Reproduction rate: 100.0000%

=== CHAIN BOUNDARIES ===
Initial seed: 0000000000000000000000000000000000000000000000000000000000000000
First reproduced hash: f944c53db8a7ef5d3cb9c4d0a82f7298351ecc8eebc014b57399c82b75408249
Last reproduced hash: 9e10c8d55bfa28fa843a697b5972881d0fa6f2161f5ef12ef1471bd31bc49b62
Last stored hash: 9e10c8d55bfa28fa843a697b5972881d0fa6f2161f5ef12ef1471bd31bc49b62

VALIDATION: PASS


In [6]:
# ============================================================
# CELL 6 — Independent Merkle root reproduction
# ============================================================
# Reproduce the original logger's Merkle tree construction.
#
# Original sources:
# solar_logger/Codes/logger.py
# https://github.com/Tierion/pymerkletools
#
# This implementation uses Python's standard library.

def sha256_bytes(value):
    """Return the binary SHA-256 digest."""
    return hashlib.sha256(value).digest()


def merkle_root(leaves):
    """Calculate a Merkle root using the original tree rules."""

    if not leaves:
        raise ValueError("Cannot construct an empty Merkle tree.")

    level = list(leaves)

    while len(level) > 1:

        next_level = []

        for index in range(0, len(level), 2):

            if index + 1 < len(level):
                # Hash the concatenated binary digests.
                parent = sha256_bytes(
                    level[index] + level[index + 1]
                )
            else:
                # Promote an unpaired node unchanged.
                parent = level[index]

            next_level.append(parent)

        level = next_level

    return level[0]


# Validate single, even and odd leaf configurations.
a = sha256_bytes(b"a")
b = sha256_bytes(b"b")
c = sha256_bytes(b"c")

assert merkle_root([a]) == a

assert merkle_root([a, b]) == sha256_bytes(a + b)

assert merkle_root([a, b, c]) == sha256_bytes(
    sha256_bytes(a + b) + c
)

# Reproduce the historical Merkle commitments.
merkle_results = []

for batch_index, batch in enumerate(batches):

    leaves = [
        sha256_bytes(
            ",".join(
                record[field] for field in EXPECTED_FIELDS
            ).encode("utf-8")
        )
        for record in batch["records"]
    ]

    computed_root = "0x" + merkle_root(leaves).hex()

    expected_root = batch["proof"]["root"].lower()

    merkle_results.append({
        "batch_index": batch_index,
        "batch_size": len(leaves),
        "expected_root": expected_root,
        "computed_root": computed_root,
        "match": computed_root == expected_root,
    })

# Summarize the reproduction results.
merkle_mismatches = [
    result for result in merkle_results
    if not result["match"]
]

matched_roots = len(merkle_results) - len(merkle_mismatches)

print("=== MERKLE IMPLEMENTATION TESTS ===")
print("Single leaf: PASS")
print("Even leaf count: PASS")
print("Odd leaf count: PASS")

print("\n=== MERKLE ROOT REPRODUCTION ===")
print("Batches checked:", len(merkle_results))
print("Matching roots:", matched_roots)
print("Mismatched roots:", len(merkle_mismatches))

print("Total leaves verified:",
      sum(result["batch_size"] for result in merkle_results))

print(f"Reproduction rate: "
      f"{matched_roots / len(merkle_results):.4%}")

print("\n=== FIRST BATCH ===")
print("Expected:", merkle_results[0]["expected_root"])
print("Computed:", merkle_results[0]["computed_root"])

print("\n=== LAST BATCH ===")
print("Expected:", merkle_results[-1]["expected_root"])
print("Computed:", merkle_results[-1]["computed_root"])

if merkle_mismatches:
    print("\nFirst five mismatches:")
    for mismatch in merkle_mismatches[:5]:
        print(mismatch)

assert len(merkle_results) == len(proofs)
assert not merkle_mismatches, "Merkle root reproduction failed."

print("\nVALIDATION: PASS")

=== MERKLE IMPLEMENTATION TESTS ===
Single leaf: PASS
Even leaf count: PASS
Odd leaf count: PASS

=== MERKLE ROOT REPRODUCTION ===
Batches checked: 130
Matching roots: 130
Mismatched roots: 0
Total leaves verified: 8085
Reproduction rate: 100.0000%

=== FIRST BATCH ===
Expected: 0x02e25c78da865ca64610a1f02fa7395287bbe46e38b33b25981050137a1ba645
Computed: 0x02e25c78da865ca64610a1f02fa7395287bbe46e38b33b25981050137a1ba645

=== LAST BATCH ===
Expected: 0x8d5bf70ceb6cae1ff94c925518ebaec8b137dadf1be52c1a43dfbe80a47dea32
Computed: 0x8d5bf70ceb6cae1ff94c925518ebaec8b137dadf1be52c1a43dfbe80a47dea32

VALIDATION: PASS


In [7]:
# ============================================================
# CELL 7 — Historical batch hash reproduction
# ============================================================
# Reconstruct the original JSON-based batch commitments.
#
# Source: original solar_logger/Codes/logger.py
# Original method: SHA256(json.dumps(queue))
#
# Preserve field order and reconstruct numerical types
# without changing the original CSV records.


def reconstruct_json_record(record):
    """Reconstruct a candidate original JSON measurement."""

    reconstructed = {}

    for field in EXPECTED_FIELDS:

        if field in NUMERIC_FIELDS:

            # Recover the numerical type from its CSV representation.
            value = json.loads(record[field])

            assert type(value) in (int, float)

            # Confirm that numerical reconstruction preserves
            # the original Python textual representation.
            assert str(value) == record[field], (
                f"Serialization mismatch: {field}"
            )

            reconstructed[field] = value

        else:
            reconstructed[field] = record[field]

    return reconstructed


# Independently reproduce each historical batch commitment.
batch_hash_results = []

for batch_index, batch in enumerate(batches):

    reconstructed_queue = [
        reconstruct_json_record(record)
        for record in batch["records"]
    ]

    # Use the original logger's default JSON serialization.
    serialized_queue = json.dumps(reconstructed_queue)

    computed_hash = "0x" + hashlib.sha256(
        serialized_queue.encode("utf-8")
    ).hexdigest()

    expected_hash = batch["proof"]["batch_hash"].lower()

    batch_hash_results.append({
        "batch_index": batch_index,
        "batch_size": len(reconstructed_queue),
        "expected_hash": expected_hash,
        "computed_hash": computed_hash,
        "match": computed_hash == expected_hash,
    })


# Evaluate reproduction results.
batch_hash_mismatches = [
    result for result in batch_hash_results
    if not result["match"]
]

matched_batch_hashes = (
    len(batch_hash_results) - len(batch_hash_mismatches)
)

print("=== BATCH HASH REPRODUCTION ===")
print("Batches checked:", len(batch_hash_results))
print("Matching batch hashes:", matched_batch_hashes)
print("Mismatched batch hashes:", len(batch_hash_mismatches))

print(f"Reproduction rate: "
      f"{matched_batch_hashes / len(batch_hash_results):.4%}")

print("\n=== FIRST BATCH ===")
print("Expected:", batch_hash_results[0]["expected_hash"])
print("Computed:", batch_hash_results[0]["computed_hash"])

print("\n=== LAST BATCH ===")
print("Expected:", batch_hash_results[-1]["expected_hash"])
print("Computed:", batch_hash_results[-1]["computed_hash"])

if batch_hash_mismatches:
    print("\nFirst five mismatches:")
    for mismatch in batch_hash_mismatches[:5]:
        print(mismatch)

assert len(batch_hash_results) == len(proofs)
assert not batch_hash_mismatches, "Batch hash reproduction failed."

print("\nVALIDATION: PASS")

=== BATCH HASH REPRODUCTION ===
Batches checked: 130
Matching batch hashes: 130
Mismatched batch hashes: 0
Reproduction rate: 100.0000%

=== FIRST BATCH ===
Expected: 0x624e15549651527f09e0e9f29b9d5389cd83e4832ca5d5cfb3f8ae9f99a057e9
Computed: 0x624e15549651527f09e0e9f29b9d5389cd83e4832ca5d5cfb3f8ae9f99a057e9

=== LAST BATCH ===
Expected: 0x5450850cec3c355d16c31108a9bf253fb14ef7c2bfd49ab590373853a68d48f5
Computed: 0x5450850cec3c355d16c31108a9bf253fb14ef7c2bfd49ab590373853a68d48f5

VALIDATION: PASS


In [9]:
# ============================================================
# CELL 8 — Historical Ethereum transaction receipts
# ============================================================
# Retrieve and preserve historical transaction receipts.
# Read-only RPC requests. No wallet or private key required.

from urllib.request import Request, urlopen
from datetime import timezone
import time

RPC_URL = "https://sepolia.gateway.tenderly.co"
SEPOLIA_CHAIN_ID = 11155111

# Original SolarLoggerV3 deployment address on Sepolia.
CONTRACT_ADDRESS = "0xc986b14f1d8a26fb46b10d6fda35c5d4062bda98"

RECEIPT_FILE = DATA / "sepolia_receipts.json"


def rpc_request(method, params=None):
    """Execute a read-only Ethereum JSON-RPC request."""

    payload = {
        "jsonrpc": "2.0",
        "method": method,
        "params": params or [],
        "id": 1,
    }

    request = Request(
        RPC_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )

    with urlopen(request, timeout=30) as response:
        result = json.load(response)

    if "error" in result:
        raise RuntimeError(result["error"])

    return result["result"]


# Reuse preserved evidence when available.
new_snapshot = not RECEIPT_FILE.exists()

if not new_snapshot:

    snapshot = json.loads(RECEIPT_FILE.read_text())
    print("Using preserved transaction receipts.")

else:

    chain_id = int(rpc_request("eth_chainId"), 16)
    assert chain_id == SEPOLIA_CHAIN_ID

    receipts = []

    print("=== RETRIEVING TRANSACTION RECEIPTS ===")

    for index, proof in enumerate(proofs, start=1):

        receipt = rpc_request(
            "eth_getTransactionReceipt",
            [proof["tx_hash"]],
        )

        assert receipt is not None, (
            f"Missing receipt: {proof['tx_hash']}"
        )

        receipts.append(receipt)

        if index % 10 == 0 or index == len(proofs):
            print(f"Retrieved: {index}/{len(proofs)}")

        # Operational pause to limit request frequency.
        time.sleep(0.2)

    snapshot = {
        "source_commit": SOURCE_COMMIT,
        "provider": RPC_URL,
        "chain_id": chain_id,
        "retrieved_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "receipts": receipts,
    }


# Validate source provenance and receipt inventory.
assert snapshot["source_commit"] == SOURCE_COMMIT
assert snapshot["chain_id"] == SEPOLIA_CHAIN_ID

receipts = snapshot["receipts"]

assert len(receipts) == len(proofs)

successful_transactions = 0
correct_recipients = 0
matching_transactions = 0

for proof, receipt in zip(proofs, receipts):

    assert receipt["transactionHash"].lower() == (
        proof["tx_hash"].lower()
    )

    matching_transactions += 1

    assert int(receipt["status"], 16) == 1
    successful_transactions += 1

    assert receipt["to"].lower() == CONTRACT_ADDRESS
    correct_recipients += 1

    assert receipt["blockHash"]
    assert int(receipt["blockNumber"], 16) > 0


# Preserve the complete snapshot only after validation.
if new_snapshot:

    with RECEIPT_FILE.open("x", encoding="utf-8") as stream:
        json.dump(snapshot, stream, indent=2)


block_numbers = [
    int(receipt["blockNumber"], 16)
    for receipt in receipts
]

print("\n=== HISTORICAL TRANSACTION AUDIT ===")
print("Provider:", snapshot["provider"])
print("Chain ID:", snapshot["chain_id"])
print("Receipts:", len(receipts))
print("Matching transaction hashes:", matching_transactions)
print("Successful transactions:", successful_transactions)
print("Correct contract recipients:", correct_recipients)
print("Total event logs:", sum(
    len(receipt["logs"]) for receipt in receipts
))
print("Unique blocks:", len(set(block_numbers)))
print("First block:", min(block_numbers))
print("Last block:", max(block_numbers))

print("\n=== EVIDENCE PROVENANCE ===")
print("Retrieved at:", snapshot["retrieved_at_utc"])
print("Snapshot:", RECEIPT_FILE.relative_to(ROOT))
print("SHA-256:", hashlib.sha256(
    RECEIPT_FILE.read_bytes()
).hexdigest())

print("\nVALIDATION: PASS")

=== RETRIEVING TRANSACTION RECEIPTS ===
Retrieved: 10/130
Retrieved: 20/130
Retrieved: 30/130
Retrieved: 40/130
Retrieved: 50/130
Retrieved: 60/130
Retrieved: 70/130
Retrieved: 80/130
Retrieved: 90/130
Retrieved: 100/130
Retrieved: 110/130
Retrieved: 120/130
Retrieved: 130/130

=== HISTORICAL TRANSACTION AUDIT ===
Provider: https://sepolia.gateway.tenderly.co
Chain ID: 11155111
Receipts: 130
Matching transaction hashes: 130
Successful transactions: 130
Correct contract recipients: 130
Total event logs: 130
Unique blocks: 130
First block: 8537222
Last block: 8574745

=== EVIDENCE PROVENANCE ===
Retrieved at: 2026-11-24T19:48:32.348883+00:00
Snapshot: data/sepolia_receipts.json
SHA-256: 7579da1657c8d84cda4ecfbbcec6ea9ac0d5686536bfc2fbe2cbc59de84568a3

VALIDATION: PASS


In [13]:
# ============================================================
# CELL 9 — Historical smart contract event verification
# ============================================================
# Verify historical Ethereum event commitments.
# Preserve observed timestamp discrepancies.
# Test the timezone-offset hypothesis separately.
#
# References:
# Original solar_logger/Codes/logger.py
# Original solar_logger/Codes/smart_contract.sol
# Solidity ABI specification: Events
# Python 3.12 documentation: datetime.timestamp()

from eth_abi import decode
from eth_hash.auto import keccak

EVENT_SIGNATURE = (
    "BatchLogged(address,uint256,bytes32,bytes32,"
    "bytes32,uint256,string,uint256)"
)

EVENT_TOPIC = "0x" + keccak(
    EVENT_SIGNATURE.encode("utf-8")
).hex()

EVENT_DATA_TYPES = [
    "bytes32",
    "bytes32",
    "uint256",
    "string",
    "uint256",
]

# Exploratory offset hypothesis based on observed discrepancies
# and the timezone-naive conversion in the original logger.
# This is not a verified historical timezone configuration.
CANDIDATE_OFFSET_SECONDS = 2 * 60 * 60

decoded_events = []
event_checks = []
timestamp_diagnostics = []

for index, (batch, receipt) in enumerate(
    zip(batches, receipts, strict=True)
):

    proof = batch["proof"]

    matching_logs = [
        log for log in receipt["logs"]
        if log["address"].lower() == CONTRACT_ADDRESS
        and log["topics"]
        and log["topics"][0].lower() == EVENT_TOPIC
    ]

    assert len(matching_logs) == 1

    log = matching_logs[0]
    topics = log["topics"]

    assert len(topics) == 4

    submitter = decode(
        ["address"], bytes.fromhex(topics[1][2:])
    )[0]

    event_index = decode(
        ["uint256"], bytes.fromhex(topics[2][2:])
    )[0]

    device_hash = topics[3].lower()

    (
        event_root,
        event_batch_hash,
        event_timestamp,
        event_device_id,
        event_energy,
    ) = decode(
        EVENT_DATA_TYPES,
        bytes.fromhex(log["data"][2:])
    )

    event_root = "0x" + event_root.hex()
    event_batch_hash = "0x" + event_batch_hash.hex()

    expected_device_ids = {
        record["device_id"]
        for record in batch["records"]
    }

    expected_device_hash = "0x" + keccak(
        event_device_id.encode("utf-8")
    ).hex()

    # Interpret measurement timestamps according to the
    # original logger's UTC measurement convention.
    last_measurement = datetime.fromisoformat(
        batch["records"][-1]["timestamp"]
    ).replace(tzinfo=timezone.utc)

    proof_time = datetime.fromisoformat(proof["timestamp"])

    measurement_epoch = int(last_measurement.timestamp())
    proof_epoch = int(proof_time.timestamp())

    # Preserve the original temporal validation criterion.
    raw_timestamp_valid = (
        measurement_epoch <= event_timestamp <= proof_epoch
    )

    # Test the candidate offset without modifying source data.
    adjusted_timestamp = (
        event_timestamp + CANDIDATE_OFFSET_SECONDS
    )

    adjusted_timestamp_valid = (
        measurement_epoch <= adjusted_timestamp <= proof_epoch
    )

    timestamp_diagnostics.append({
        "batch_index": index,
        "raw_event_timestamp": event_timestamp,
        "event_minus_measurement": (
            event_timestamp - measurement_epoch
        ),
        "candidate_adjusted_timestamp": adjusted_timestamp,
        "raw_timestamp_valid": raw_timestamp_valid,
        "adjusted_timestamp_valid": adjusted_timestamp_valid,
        "minimum_offset": measurement_epoch - event_timestamp,
        "maximum_offset": proof_epoch - event_timestamp,
    })

    checks = {
        "merkle_root": (
            event_root == merkle_results[index]["computed_root"]
        ),
        "batch_hash": (
            event_batch_hash
            == batch_hash_results[index]["computed_hash"]
        ),
        "submitter": (
            submitter.lower() == receipt["from"].lower()
        ),
        "device_id": (
            expected_device_ids == {event_device_id}
        ),
        "device_hash": (
            device_hash == expected_device_hash
        ),
        "cumulative_energy": (
            event_energy == proof["cumulative_energy_mWh"]
        ),
        "timestamp": raw_timestamp_valid,
        "transaction_hash": (
            log["transactionHash"].lower()
            == proof["tx_hash"].lower()
        ),
        "block_hash": (
            log["blockHash"].lower()
            == receipt["blockHash"].lower()
        ),
    }

    event_checks.append(checks)

    decoded_events.append({
        "batch_index": index,
        "contract_index": event_index,
        "transaction_hash": proof["tx_hash"],
        "submitter": submitter,
        "device_id": event_device_id,
        "device_hash": device_hash,
        "merkle_root": event_root,
        "batch_hash": event_batch_hash,
        "timestamp": event_timestamp,
        "cumulative_energy_mWh": event_energy,
        "checks": checks,
    })

# Report all original verification results.
print("=== SMART CONTRACT EVENT VERIFICATION ===")
print("Events decoded:", len(decoded_events))

for field in event_checks[0]:
    matches = sum(checks[field] for checks in event_checks)
    print(f"{field}: {matches}/{len(event_checks)}")

# Verify contract history consistency.
contract_indices = [
    event["contract_index"]
    for event in decoded_events
]

unique_indices = (
    len(set(contract_indices)) == len(contract_indices)
)

increasing_indices = all(
    current > previous
    for previous, current in zip(
        contract_indices, contract_indices[1:]
    )
)

print("\n=== CONTRACT HISTORY ===")
print("Unique indices:", unique_indices)
print("Increasing indices:", increasing_indices)
print("First index:", contract_indices[0])
print("Last index:", contract_indices[-1])
print("Unique submitters:", len({
    event["submitter"] for event in decoded_events
}))

# Analyze temporal discrepancies separately.
raw_valid_count = sum(
    item["raw_timestamp_valid"]
    for item in timestamp_diagnostics
)

adjusted_valid_count = sum(
    item["adjusted_timestamp_valid"]
    for item in timestamp_diagnostics
)

offset_distribution = Counter(
    item["event_minus_measurement"]
    for item in timestamp_diagnostics
)

print("\n=== TEMPORAL ASSURANCE ===")
print("Raw timestamps consistent:", raw_valid_count)
print("Raw timestamp failures:",
      len(timestamp_diagnostics) - raw_valid_count)
print("Offset distribution:", dict(offset_distribution))

print("\n=== EXPLORATORY TIMEZONE HYPOTHESIS ===")
print("Candidate offset:", CANDIDATE_OFFSET_SECONDS, "seconds")
print("Consistent after candidate adjustment:", adjusted_valid_count)

print("Common feasible offset interval:", (
    max(item["minimum_offset"] for item in timestamp_diagnostics),
    min(item["maximum_offset"] for item in timestamp_diagnostics),
))

# Preserve all failed checks, including timestamps.
failed_checks = [
    (index, field)
    for index, checks in enumerate(event_checks)
    for field, passed in checks.items()
    if not passed
]

non_temporal_failures = [
    failure for failure in failed_checks
    if failure[1] != "timestamp"
]

print("\n=== FINAL AUDIT ===")
print("Total original failed checks:", len(failed_checks))
print("Non-temporal failed checks:", len(non_temporal_failures))
print("Cryptographic commitments checked:", len(decoded_events) * 2)

assert len(decoded_events) == len(proofs)
assert unique_indices
assert increasing_indices
assert not non_temporal_failures

print("\nCOMMITMENT VALIDATION: PASS")

print("RAW TEMPORAL VALIDATION:",
      "PASS" if raw_valid_count == len(proofs) else "FAIL")

print("TIMEZONE HYPOTHESIS:",
      "CONSISTENT" if adjusted_valid_count == len(proofs)
      else "NOT FULLY CONSISTENT")

print("HISTORICAL DEVICE TIMEZONE: UNVERIFIED")

=== SMART CONTRACT EVENT VERIFICATION ===
Events decoded: 130
merkle_root: 130/130
batch_hash: 130/130
submitter: 130/130
device_id: 130/130
device_hash: 130/130
cumulative_energy: 130/130
timestamp: 0/130
transaction_hash: 130/130
block_hash: 130/130

=== CONTRACT HISTORY ===
Unique indices: True
Increasing indices: True
First index: 174
Last index: 303
Unique submitters: 1

=== TEMPORAL ASSURANCE ===
Raw timestamps consistent: 0
Raw timestamp failures: 130
Offset distribution: {-7200: 126, -7199: 4}

=== EXPLORATORY TIMEZONE HYPOTHESIS ===
Candidate offset: 7200 seconds
Consistent after candidate adjustment: 130
Common feasible offset interval: (7200, 7201)

=== FINAL AUDIT ===
Total original failed checks: 130
Non-temporal failed checks: 0
Cryptographic commitments checked: 260

COMMITMENT VALIDATION: PASS
RAW TEMPORAL VALIDATION: FAIL
TIMEZONE HYPOTHESIS: CONSISTENT
HISTORICAL DEVICE TIMEZONE: UNVERIFIED


In [14]:
# ============================================================
# CELL 10 — Ethereum block timestamp verification
# ============================================================
# Retrieve block evidence and compare blockchain timestamps
# against original measurement and application timestamps.
#
# Reference: Ethereum JSON-RPC, eth_getBlockByHash.

BLOCK_FILE = DATA / "sepolia_blocks.json"

receipt_snapshot_hash = hashlib.sha256(
    RECEIPT_FILE.read_bytes()
).hexdigest()

new_block_snapshot = not BLOCK_FILE.exists()

if new_block_snapshot:

    assert int(rpc_request("eth_chainId"), 16) == SEPOLIA_CHAIN_ID

    blocks = {}

    print("=== RETRIEVING ETHEREUM BLOCKS ===")

    for index, receipt in enumerate(receipts, start=1):

        block_hash = receipt["blockHash"].lower()

        if block_hash not in blocks:

            block = rpc_request(
                "eth_getBlockByHash",
                [block_hash, False],
            )

            assert block is not None, (
                f"Block unavailable: {block_hash}"
            )

            blocks[block_hash] = block

            time.sleep(0.2)

        if index % 10 == 0 or index == len(receipts):
            print(f"Processed: {index}/{len(receipts)}")

    block_snapshot = {
        "source_commit": SOURCE_COMMIT,
        "receipt_snapshot_sha256": receipt_snapshot_hash,
        "provider": RPC_URL,
        "chain_id": SEPOLIA_CHAIN_ID,
        "retrieved_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "blocks": blocks,
    }

else:

    block_snapshot = json.loads(BLOCK_FILE.read_text())
    print("Using preserved Ethereum blocks.")


# Validate the provenance and inventory of block evidence.
assert block_snapshot["source_commit"] == SOURCE_COMMIT
assert block_snapshot["chain_id"] == SEPOLIA_CHAIN_ID
assert block_snapshot["receipt_snapshot_sha256"] == (
    receipt_snapshot_hash
)

blocks = block_snapshot["blocks"]

assert set(blocks) == {
    receipt["blockHash"].lower()
    for receipt in receipts
}

block_diagnostics = []

for batch, receipt, event in zip(
    batches, receipts, decoded_events, strict=True
):

    block = blocks[receipt["blockHash"].lower()]

    # Check consistency between receipt and block metadata.
    assert block["hash"].lower() == (
        receipt["blockHash"].lower()
    )

    assert int(block["number"], 16) == (
        int(receipt["blockNumber"], 16)
    )

    # Confirm transaction membership and position.
    transaction_index = int(receipt["transactionIndex"], 16)

    assert 0 <= transaction_index < len(block["transactions"])

    assert block["transactions"][transaction_index].lower() == (
        receipt["transactionHash"].lower()
    )

    block_epoch = int(block["timestamp"], 16)

    measurement_epoch = int(
        datetime.fromisoformat(
            batch["records"][-1]["timestamp"]
        ).replace(tzinfo=timezone.utc).timestamp()
    )

    proof_epoch = int(
        datetime.fromisoformat(
            batch["proof"]["timestamp"]
        ).timestamp()
    )

    event_epoch = event["timestamp"]

    adjusted_epoch = (
        event_epoch + CANDIDATE_OFFSET_SECONDS
    )

    block_diagnostics.append({
        "block_number": int(block["number"], 16),
        "block_timestamp": block_epoch,
        "block_minus_measurement": (
            block_epoch - measurement_epoch
        ),
        "block_minus_raw_event": block_epoch - event_epoch,
        "block_minus_adjusted_event": (
            block_epoch - adjusted_epoch
        ),
        "proof_minus_block": proof_epoch - block_epoch,
    })


# Preserve the retrieved evidence without modifying raw data.
if new_block_snapshot:

    with BLOCK_FILE.open("x", encoding="utf-8") as stream:
        json.dump(block_snapshot, stream, indent=2)


print("\n=== BLOCK EVIDENCE AUDIT ===")
print("Unique blocks:", len(blocks))
print("Transaction membership checks:", len(block_diagnostics))
print("Block hash and number checks: PASS")

print("\n=== TEMPORAL DIFFERENCES (SECONDS) ===")

for field in (
    "block_minus_measurement",
    "block_minus_raw_event",
    "block_minus_adjusted_event",
    "proof_minus_block",
):

    values = [
        item[field] for item in block_diagnostics
    ]

    print(f"\n{field}")
    print("Minimum:", min(values))
    print("Median:", median(values))
    print("Maximum:", max(values))
    print("Negative values:", sum(v < 0 for v in values))


print("\n=== FIRST BATCH ===")

first = block_diagnostics[0]

print("Block number:", first["block_number"])

print("Block timestamp:", datetime.fromtimestamp(
    first["block_timestamp"], tz=timezone.utc
).isoformat())

print("Block minus raw event:",
      first["block_minus_raw_event"])

print("Block minus adjusted event:",
      first["block_minus_adjusted_event"])


print("\n=== EVIDENCE PROVENANCE ===")
print("Provider:", block_snapshot["provider"])
print("Retrieved at:", block_snapshot["retrieved_at_utc"])
print("Snapshot:", BLOCK_FILE.relative_to(ROOT))
print("SHA-256:", hashlib.sha256(
    BLOCK_FILE.read_bytes()
).hexdigest())

print("\nBLOCK EVIDENCE CONSISTENCY: PASS")
print("TEMPORAL DIAGNOSTICS: COMPLETE")

=== RETRIEVING ETHEREUM BLOCKS ===
Processed: 10/130
Processed: 20/130
Processed: 30/130
Processed: 40/130
Processed: 50/130
Processed: 60/130
Processed: 70/130
Processed: 80/130
Processed: 90/130
Processed: 100/130
Processed: 110/130
Processed: 120/130
Processed: 130/130

=== BLOCK EVIDENCE AUDIT ===
Unique blocks: 130
Transaction membership checks: 130
Block hash and number checks: PASS

=== TEMPORAL DIFFERENCES (SECONDS) ===

block_minus_measurement
Minimum: 11
Median: 12.0
Maximum: 12
Negative values: 0

block_minus_raw_event
Minimum: 7211
Median: 7212.0
Maximum: 7212
Negative values: 0

block_minus_adjusted_event
Minimum: 11
Median: 12.0
Maximum: 12
Negative values: 0

proof_minus_block
Minimum: -10
Median: -10.0
Maximum: -3
Negative values: 130

=== FIRST BATCH ===
Block number: 8537222
Block timestamp: 2025-06-13T03:57:24+00:00
Block minus raw event: 7212
Block minus adjusted event: 12

=== EVIDENCE PROVENANCE ===
Provider: https://sepolia.gateway.tenderly.co
Retrieved at: 2026-

## Historical Baseline: Findings and Limitations

### Verified results

- Stored measurement rows: 10,268.
- Unique measurements: 8,106.
- Duplicate rollover rows: 2,162.
- Observation duration: 135.083636 hours.
- Unsubmitted measurements: 21.
- Reproduced record hashes: 8,106/8,106.
- Reproduced Merkle roots: 130/130.
- Reproduced JSON batch hashes: 130/130.
- Successful transaction receipts: 130/130.
- Matching event commitments: 260/260.
- Transaction membership checks: 130/130.

### Observed temporal discrepancy

All 130 application-generated event timestamps
failed the original temporal consistency criterion.

126 events preceded the final batch measurement
by 7,200 seconds, and four by 7,199 seconds.

A candidate adjustment of 7,200 seconds reconciled
all events with the original temporal bounds.

The discrepancy is consistent with timezone-naive
datetime conversion in the original logger.

The historical device timezone remains unverified.
No original timestamps were modified.

### Block timestamp corroboration

Block timestamps followed final measurements
by 11-12 seconds.

Local proof timestamps preceded block timestamps
by 3-10 seconds.

The original logger returns the transaction hash
without waiting for a transaction receipt.
This explains why local proof timestamps can
precede block inclusion.

### Scientific implications

Cryptographic commitment consistency does not
establish temporal or semantic validity.

The observed discrepancy provides an empirical
motivation for independent evidence-validation rules.

### Limitations

- Single-device experimental dataset.
- Physical measurement accuracy was not established.
- Historical timezone configuration remains unverified.
- Ethereum evidence relies on one external RPC provider.
- Consensus and cryptographic inclusion proofs were
  not independently verified.
- The timezone hypothesis was developed retrospectively.
- Historical findings are not blinded benchmark results.

### Conclusion

The historical cryptographic baseline is reproduced
successfully, subject to the documented limitations.

Raw temporal validation failed and remains an
explicit empirical finding.

The baseline is suitable for controlled comparative
benchmark development.